# Data Analytics Lab Sheet 8: Customer Churn Prediction using Decision Tree Classification

**Name:** ANURAG PARMAR  
**Roll No.:** 8  
**Section:** B  
**CU ID:** CU24250076  
**Program:** B.Tech CSE  

## Aim
Train and evaluate a Decision Tree classifier using a reproducible synthetic customer churn dataset.

> Run cells from top to bottom. Dataset sources and any synthetic-data notes are documented in the relevant sections.

## 1. Import libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score, classification_report
sns.set_theme(style="whitegrid")

## 2. Generate customer churn data

In [ ]:
rng = np.random.default_rng(42)
n = 1000
age = rng.integers(18, 76, n)
tenure = rng.integers(1, 73, n)
monthly_charges = np.round(rng.uniform(20, 130, n), 2)
support_calls = rng.poisson(2.2, n).clip(0, 10)
contract_months = rng.choice([1, 12, 24], size=n, p=[0.50, 0.30, 0.20])
late_payments = rng.poisson(1.0, n).clip(0, 6)
risk = (-2.0 + 0.025 * (age - 40) - 0.025 * tenure
        + 0.025 * (monthly_charges - 70) + 0.30 * support_calls
        - 0.035 * contract_months + 0.25 * late_payments)
probability = 1 / (1 + np.exp(-risk))
churn = rng.binomial(1, np.clip(probability, 0.03, 0.95))
df = pd.DataFrame({
    "Age": age, "Tenure": tenure, "MonthlyCharges": monthly_charges,
    "SupportCalls": support_calls, "ContractMonths": contract_months,
    "LatePayments": late_payments, "Churn": churn
})
df["Churn"] = df["Churn"].map({0: "No", 1: "Yes"})
display(df.head())
display(df["Churn"].value_counts())

## 3. Explore data and visualize churn

In [ ]:
print("Missing values:")
display(df.isnull().sum())
print("Duplicate rows:", df.duplicated().sum())
df = df.drop_duplicates().copy()
sns.countplot(data=df, x="Churn")
plt.title("Customer Churn Distribution")
plt.show()
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.boxplot(data=df, x="Churn", y="MonthlyCharges", ax=axes[0])
sns.boxplot(data=df, x="Churn", y="Tenure", ax=axes[1])
plt.tight_layout()
plt.show()

## 4. Split data and train model

In [ ]:
X = df.drop(columns=["Churn"])
y = df["Churn"].map({"No": 0, "Yes": 1})
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
model = DecisionTreeClassifier(criterion="gini", max_depth=5, random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print("Training samples:", len(X_train), "Testing samples:", len(X_test))

## 5. Evaluate classification performance

In [ ]:
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["No Churn", "Churn"], yticklabels=["No Churn", "Churn"])
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()
metrics = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1 Score"],
    "Value": [
        accuracy_score(y_test, y_pred),
        precision_score(y_test, y_pred, zero_division=0),
        recall_score(y_test, y_pred, zero_division=0),
        f1_score(y_test, y_pred, zero_division=0)
    ]
})
display(metrics)
print(classification_report(y_test, y_pred, target_names=["No Churn", "Churn"], zero_division=0))

## 6. Visualize tree and feature importance

In [ ]:
plt.figure(figsize=(20, 10))
plot_tree(model, feature_names=X.columns, class_names=["No Churn", "Churn"],
          filled=True, rounded=True, fontsize=8)
plt.title("Decision Tree for Customer Churn Prediction")
plt.show()

importance = pd.DataFrame({"Feature": X.columns, "Importance": model.feature_importances_})
importance = importance.sort_values("Importance", ascending=False)
display(importance)
sns.barplot(data=importance, x="Importance", y="Feature")
plt.title("Feature Importance")
plt.tight_layout()
plt.show()
print("Top features are predictive associations, not proof of causation.")